In [1]:
import sys
import os

# Use absolute paths to ensure imports work from anywhere
pipeline_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline"
eda_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline\eda"

sys.path.insert(0, eda_dir)
sys.path.insert(0, pipeline_dir)

import polars as pl
import numpy as np
import pandas as pd
from config import *
from eda_base import *
from eda_stats import *

# Employment Density Data (S2 Output)
Aggregate employment density by commune and occupation

In [2]:
# Load commune density data for all years
import glob

density_dir = os.path.join(OUT_DIR, 'commune_density')
density_files = sorted(glob.glob(os.path.join(density_dir, '*.parquet')))

print(f"Found {len(density_files)} density files")

if density_files:
    # Load all files but handle schema mismatch using diagonal concat
    all_densities = [pl.read_parquet(f) for f in density_files]
    
    # Use diagonal_relaxed to handle different schemas
    dens = pl.concat(all_densities, how='diagonal_relaxed')
    print(f"\nShape: {dens.shape}")
    print(f"Columns: {dens.columns}")

Found 21 density files

Shape: (11035689, 52)
Columns: ['comt', 'norm', 'cs2', 'origin', 'neighbor', 'nb_jobs_cs2_10', 'nb_jobs_cs2_21', 'nb_jobs_cs2_22', 'nb_jobs_cs2_23', 'nb_jobs_cs2_31', 'nb_jobs_cs2_33', 'nb_jobs_cs2_34', 'nb_jobs_cs2_35', 'nb_jobs_cs2_37', 'nb_jobs_cs2_38', 'nb_jobs_cs2_42', 'nb_jobs_cs2_43', 'nb_jobs_cs2_44', 'nb_jobs_cs2_45', 'nb_jobs_cs2_46', 'nb_jobs_cs2_47', 'nb_jobs_cs2_48', 'nb_jobs_cs2_52', 'nb_jobs_cs2_53', 'nb_jobs_cs2_54', 'nb_jobs_cs2_55', 'nb_jobs_cs2_56', 'nb_jobs_cs2_62', 'nb_jobs_cs2_63', 'nb_jobs_cs2_64', 'nb_jobs_cs2_65', 'nb_jobs_cs2_67', 'nb_jobs_cs2_68', 'nb_jobs_cs2_69', 'nb_jobs_cs2_missing', 'nb_jobs_cs2_11', 'nb_jobs_cs2_12', 'nb_jobs_cs2_13', 'nb_jobs_cs2_71', 'nb_jobs_cs2_72', 'nb_jobs_cs2_74', 'nb_jobs_cs2_75', 'nb_jobs_cs2_77', 'nb_jobs_cs2_78', 'nb_jobs_cs2_81', 'nb_jobs_cs2_83', 'nb_jobs_cs2_84', 'nb_jobs_cs2_85', 'nb_jobs_cs2_86', 'nb_jobs_total', 'an', 'commune_has_neighbors']


In [3]:
# Temporal coverage: density across years
if 'dens' in locals() and 'an' in dens.columns:
    by_year = dens.group_by('an').agg([
        pl.col('comt').n_unique().alias('n_communes'),
        pl.col('nb_jobs_total').sum().alias('total_employment')
    ]).sort('an')
    
    print("Employment density by year:")
    print(by_year)

Employment density by year:
shape: (19, 3)
┌──────┬────────────┬──────────────────┐
│ an   ┆ n_communes ┆ total_employment │
│ ---  ┆ ---        ┆ ---              │
│ i32  ┆ u32        ┆ i64              │
╞══════╪════════════╪══════════════════╡
│ null ┆ 21688      ┆ 0                │
│ 2002 ┆ 33238      ┆ 234132964        │
│ 2003 ┆ 33238      ┆ 235894333        │
│ 2004 ┆ 33238      ┆ 237393838        │
│ 2005 ┆ 33238      ┆ 239875447        │
│ …    ┆ …          ┆ …                │
│ 2015 ┆ 33238      ┆ 251211587        │
│ 2016 ┆ 33238      ┆ 251670831        │
│ 2017 ┆ 33238      ┆ 252609605        │
│ 2018 ┆ 33238      ┆ 252640011        │
│ 2019 ┆ 33238      ┆ 253966165        │
└──────┴────────────┴──────────────────┘


In [4]:
# Geographic coverage: communes with employment
if 'dens' in locals() and 'comt' in dens.columns and 'nb_jobs_total' in dens.columns:
    commune_summary = dens.group_by('comt').agg([
        pl.col('nb_jobs_total').mean().alias('avg_employment'),
        pl.col('an').n_unique().alias('n_years_covered')
    ]).sort('avg_employment', descending=True).head(15)
    
    print("Top 15 communes by average employment:")
    print(commune_summary)

Top 15 communes by average employment:
shape: (15, 3)
┌───────┬────────────────┬─────────────────┐
│ comt  ┆ avg_employment ┆ n_years_covered │
│ ---   ┆ ---            ┆ ---             │
│ str   ┆ f64            ┆ u32             │
╞═══════╪════════════════╪═════════════════╡
│ null  ┆ null           ┆ 1               │
│ 75056 ┆ 164534.277778  ┆ 19              │
│ 92075 ┆ 164224.833333  ┆ 19              │
│ 92024 ┆ 163891.611111  ┆ 19              │
│ 92046 ┆ 163727.222222  ┆ 19              │
│ …     ┆ …              ┆ …               │
│ 92064 ┆ 162031.111111  ┆ 19              │
│ 95277 ┆ 161417.111111  ┆ 19              │
│ 92014 ┆ 161337.666667  ┆ 19              │
│ 95445 ┆ 161336.777778  ┆ 19              │
│ 92049 ┆ 161259.055556  ┆ 19              │
└───────┴────────────────┴─────────────────┘


In [5]:
# Occupation distribution: employment by CS2 codes
if 'dens' in locals():
    # Count occupation columns
    cs2_cols = [c for c in dens.columns if c.startswith('nb_jobs_cs2_')]
    print(f"Occupations (CS2 codes) covered: {len(cs2_cols)}")
    
    if cs2_cols:
        # Total employment by occupation
        occupation_totals = {}
        for col in cs2_cols:
            total = dens.select(pl.col(col).sum()).to_series()[0]
            if total is not None:
                cs2_code = col.replace('nb_jobs_cs2_', '')
                occupation_totals[cs2_code] = total
        
        # Sort by employment
        sorted_occs = sorted(occupation_totals.items(), key=lambda x: x[1], reverse=True)
        print(f"\nTop 10 occupations by total employment:")
        for cs2, jobs in sorted_occs[:10]:
            print(f"  CS2 {cs2}: {jobs:>10,.0f} jobs")

Occupations (CS2 codes) covered: 44

Top 10 occupations by total employment:
  CS2 missing: 577,642,330 jobs
  CS2 56: 479,100,229 jobs
  CS2 54: 462,375,099 jobs
  CS2 37: 346,003,622 jobs
  CS2 43: 277,740,928 jobs
  CS2 46: 276,140,918 jobs
  CS2 52: 257,880,344 jobs
  CS2 62: 235,937,893 jobs
  CS2 67: 209,974,979 jobs
  CS2 55: 181,686,158 jobs


In [6]:
# Missing values: communes with neighbors
if 'dens' in locals() and 'commune_has_neighbors' in dens.columns:
    n_with_neighbors = dens.select(pl.col('commune_has_neighbors').sum()).to_series()[0]
    total = len(dens)
    pct = n_with_neighbors / total * 100
    
    print(f"Communes with neighbors: {n_with_neighbors:,} ({pct:.1f}%)")
    print(f"Communes without neighbors: {total - n_with_neighbors:,} ({100-pct:.1f}%)")

Communes with neighbors: 598,284 (5.4%)
Communes without neighbors: 10,437,405 (94.6%)


In [7]:
# Concentration: Are jobs concentrated in a few communes?
if 'dens' in locals() and 'nb_jobs_total' in dens.columns:
    # Total employment
    total_jobs = dens.select(pl.col('nb_jobs_total').sum()).to_series()[0]
    
    # Employment by commune (aggregate across years)
    commune_jobs = dens.group_by('comt').agg(
        pl.col('nb_jobs_total').sum().alias('total_jobs')
    ).sort('total_jobs', descending=True)
    
    # Top 10 communes
    top10_jobs = commune_jobs.head(10).select(pl.col('total_jobs').sum()).to_series()[0]
    top10_share = top10_jobs / total_jobs * 100
    
    print(f"Total employment across all communes-years: {total_jobs:,.0f}")
    print(f"Top 10 communes: {top10_share:.1f}% of total employment")
    
    # Number of communes
    n_communes = commune_jobs.select(pl.col('total_jobs').count()).to_series()[0]
    print(f"Number of communes: {n_communes:,}")

Total employment across all communes-years: 4,437,583,565
Top 10 communes: 0.7% of total employment
Number of communes: 33,239


In [8]:
# Data quality: zero employment communes
if 'dens' in locals() and 'nb_jobs_total' in dens.columns:
    zero_emp = dens.select((pl.col('nb_jobs_total') == 0).sum()).to_series()[0]
    null_emp = dens.select(pl.col('nb_jobs_total').is_null().sum()).to_series()[0]
    total = len(dens)
    
    print(f"Records with zero employment: {zero_emp} ({zero_emp/total*100:.1f}%)")
    print(f"Records with missing employment: {null_emp} ({null_emp/total*100:.1f}%)")
    print(f"Records with valid employment: {total - zero_emp - null_emp} ({(total-zero_emp-null_emp)/total*100:.1f}%)")

Records with zero employment: 0 (0.0%)
Records with missing employment: 10437405 (94.6%)
Records with valid employment: 598284 (5.4%)
